# Skill Extraction from Job Descriptions

Regex-matches a curated skill/tool vocabulary (BI, AI, cloud) against
posting descriptions and outputs a long-format (url, skill) table used by
the skill co-occurrence network analysis.

**Input:** `data/mia_postings_final2_fixed2.csv`
**Output:** `data/mia_skills_long.csv`

In [ ]:
import pandas as pd
import re

In [ ]:
df = pd.read_csv('data/mia_postings_final2_fixed2.csv')

In [ ]:
# Skill vocabulary, grouped by category. Patterns use word boundaries and
# negative lookaheads where a term is ambiguous on its own — e.g. 'ml' only
# counts near 'model'/'pipeline'/'engineer', 'Claude' excludes 'Claude
# University' (a real company name that otherwise false-positives), and
# 'mcp'/'rag' require nearby disambiguating words since both are common
# short acronyms with unrelated meanings.
skills = {
    # BI / Analytics
    'Power BI': r'\bpower\s*bi\b',
    'Tableau': r'\btableau\b',
    'SQL': r'\bsql\b',
    'Python': r'\bpython\b',
    'Excel': r'\bexcel\b',
    'Salesforce': r'\bsalesforce\b',
    'HubSpot': r'\bhubspot\b',
    'Looker': r'\blooker\b',
    'Google Analytics': r'\bgoogle analytics\b|\bga4\b',
    'Adobe Analytics': r'\badobe analytics\b',
    'SAS': r'\bsas\b',
    'SPSS': r'\bspss\b',
    'SAP': r'\bsap\b',
    'Google Ads': r'\bgoogle ads\b',
    'Meta Ads / Facebook Ads': r'\bmeta ads\b|\bfacebook ads\b',
    'Marketo': r'\bmarketo\b',
    'Mixpanel': r'\bmixpanel\b',
    'Amplitude': r'\bamplitude\b',

    # AI / Agentic workflow
    'ChatGPT / GPT': r'\bchatgpt\b|\bgpt-?[0-9]\b',
    'Claude': r'\bclaude\b(?!\s*(university|university\'s))',
    'Copilot': r'\bcopilot\b',
    'Gemini': r'\bgemini\b',
    'LLM': r'\bllm(s)?\b|\blarge language model',
    'Machine Learning': r'\bmachine learning\b|\bml\b(?=.{0,20}(model|pipeline|engineer))',
    'Generative AI': r'\bgenerative ai\b|\bgenai\b',
    'RAG': r'\brag\b(?=.{0,20}(pipeline|retrieval|architecture))|retrieval[- ]augmented generation',
    'Agentic / AI Agents': r'\bagentic\b|\bai agent(s)?\b|\bautonomous agent(s)?\b',
    'Prompt Engineering': r'\bprompt engineering\b',
    'LangChain': r'\blangchain\b',
    'Hugging Face': r'\bhugging\s*face\b',
    'MCP (Model Context Protocol)': r'\bmodel context protocol\b|\bmcp\b(?=.{0,20}(protocol|server|agent))',
    'AI Workflow Automation': r'\bai[- ]driven workflow|\bworkflow automation\b',

    # Cloud / Database
    'AWS': r'\baws\b|\bamazon web services\b',
    'Azure': r'\bazure\b',
    'Google Cloud (GCP)': r'\bgoogle cloud\b|\bgcp\b',
    'Snowflake': r'\bsnowflake\b',
    'BigQuery': r'\bbigquery\b',
    'Databricks': r'\bdatabricks\b',
    'Redshift': r'\bredshift\b',
    'MongoDB': r'\bmongodb\b',
    'PostgreSQL': r'\bpostgresql\b|\bpostgres\b',
    'Kubernetes': r'\bkubernetes\b|\bk8s\b',
}

In [ ]:
rows = []
for name, pattern in skills.items():
    rx = re.compile(pattern, re.IGNORECASE)
    mask = df['description'].str.contains(rx, na=False, regex=True)
    matched = df.loc[mask, ['url']].copy()
    matched['skill'] = name
    rows.append(matched)

In [ ]:
long_df = pd.concat(rows, ignore_index=True)

In [ ]:
print(long_df['skill'].value_counts())
long_df.to_csv('data/mia_skills_long.csv', index=False)

In [ ]:
# Manual QA check: spot-check 'Claude' matches for false positives
# (e.g. Claude University, a real employer name that could otherwise
# collide with the Claude LLM match).
claude_matches = df[df['description'].str.contains(r'\bclaude\b', case=False, na=False, regex=True)]
for desc in claude_matches['description'].head(10):
    idx = desc.lower().find('claude')
    print(desc[max(0,idx-60):idx+60])
    print('---')